# 🔬 RedPitayaSTCL: Laser Locking Workflow

**Scanning Transfer Cavity Lock (STCL) using RedPitaya STEMlab 125-14**

This notebook runs the complete locking sequence in clearly separated phases.
Work through each phase in order. Each phase includes a description of what to
expect and what to check before proceeding to the next.

---

### Board roles

| Label | Mode | Role | Outputs |
|-------|------|------|---------|
| `Cav` | `scan` | Scans cavity length via piezo ramp, acquires transmission signal | OUT2 → ramp, OUT1 → trigger |
| `Lock1` | `lock` | Applies PID feedback to laser current modulation inputs | OUT1 → Slave1, OUT2 → Slave2 |
| `Mon` | `monitor` | Passive cavity signal monitor for live display | — |

<br>

<blockquote style="border-left:4px solid #e67e22; padding:6px 12px;
  background:#fdf6ec; color:#7f4f00; border-radius:4px;">
  <strong> → Running with fewer boards:</strong> The notebook handles 1, 2, or 3 boards. Cells for unavailable boards are clearly marked and safe to skip.
</blockquote>

---

### Quick reference: scan timing

With default decimation `dec = 16`:
- Sample rate: `125 MHz / 16 = 7.8 MHz`
- Buffer: `2¹⁴ = 16384` samples
- Scan duration: `≈ 2.1 ms` per cycle
- Time axis: `0 → 2.1 ms` in the cavity signal plot

Range and lockpoint values are always in **milliseconds** on the PC side.
The RP converts them to buffer indices internally.


---
## Phase 0: Configuration

<blockquote style="border-left:4px solid #e67e22; padding:6px 12px;
  background:#fdf6ec; color:#7f4f00; border-radius:4px;">
  <strong> → Edit this cell only </strong> before running the notebook. All parameters are passed through to the relevant phases automatically.
</blockquote>



### Board IPs
Comment out boards that are not physically present.

### Lock parameters
Pre-filled from `settings/Default.json`. Tune these after observing the cavity signal.

**Cavity range format:** `[[r1_start, r1_end], [r2_start, r2_end]]` in ms.
Two ranges required: one for each reference peak (used to measure FSR).

**Slave range format:** `[start, end]` in ms. One range per laser channel.

**PID limit:** Output voltage clamp. Keep at `[-0.99, 0.99]` initially
(maps to ±0.99 V on the output SMA). Widen only after verifying sign and stability.


In [1]:
# ── Board addresses ────────────────────────────────────────────────────────────
RP_CAV_IP   = "192.168.0.200"   # Cav  — cavity scan RP  (required)
# RP_LOCK1_IP = "192.168.0.102" # Lock1 — laser lock RP   (comment out if absent)
# RP_MON_IP   = "192.168.0.100" # Mon   — monitor RP      (comment out if absent)

SSH_USER = "root"
SSH_PASS = "root"

In [2]:
# ── Decimation ─────────────────────────────────────────────────────────────────
# Power of 2 between 1 and 512. Higher = slower scan, more averaging.
# Default 16 gives ~2.1 ms scan duration. Change with Lock.set_dec("Cav", dec).
CAV_DEC = 16

In [3]:
# ── Cavity (Master) lock parameters ───────────────────────────────────────────
# Two ranges: one per reference peak. Values in ms.
CAV_RANGE     = [[0.25, 0.50], [1.70, 2.00]]
CAV_LOCKPOINT = 1.80   # ms — target position of the first reference peak
CAV_PID       = {"P": 0.0, "I": 2.0, "D": 0.0,
                 "I_val": 0, "limit": [-0.99, 0.99]}

In [4]:
# ── Slave 1 lock parameters (Lock1 OUT1) ──────────────────────────────────────
SL1_LABEL     = "Laser_A"
SL1_RANGE     = [0.85, 1.10]   # ms
SL1_LOCKPOINT = 0.96           # ms
SL1_ENABLED   = True
SL1_PID       = {"P": 0.0, "I": 0.5, "D": 0.0,
                 "I_val": 0, "limit": [-0.99, 0.99]}

In [5]:
# ── Slave 2 lock parameters (Lock1 OUT2) ──────────────────────────────────────
SL2_LABEL     = "Laser_B"
SL2_RANGE     = [0.50, 0.85]   # ms
SL2_LOCKPOINT = 0.60           # ms
SL2_ENABLED   = False          # set True when second laser is coupled in
SL2_PID       = {"P": 0.0, "I": 0.5, "D": 0.0,
                 "I_val": 0, "limit": [-0.99, 0.99]}

In [6]:
print("Configuration loaded.")
print("  Cav  :", RP_CAV_IP)
# print("  Lock1:", RP_LOCK1_IP)
# print("  Mon  :", RP_MON_IP)

Configuration loaded.
  Cav  : 192.168.0.200


---
## Phase 1: Upload & Connect

**What this does:**
1. Adds the repo root to `sys.path` so all imports resolve.
2. Instantiates `LockClient` — this immediately SSHes into each board,
   uploads the four RP-side scripts (`RP_Lock.py`, `RunLock.py`,
   `libserver.py`, `peak_finders.py`) via SFTP, and loads settings from
   `settings/*.json` (creating defaults if the files don't exist yet).
3. SSHes back in to start `RunLock.py` on each board, then waits 5 s for
   the Python interpreter and FPGA overlay to initialise.
4. Starts the PC-side selector event loop in a background thread.

> **If this cell hangs beyond ~30 s:** one board failed to start `RunLock.py`.
> SSH in manually and run `python3 /home/jupyter/RedPitaya/RunLock.py` to see
> the error. Common causes: port 5000 already in use (`pkill -f RunLock.py`),
> missing FPGA overlay, or wrong Python path.

> **If you see `KeyError: 'Compat'` from paramiko:** the SSH key-exchange
> negotiation failed against Ubuntu 16.04. This is fixed in `communication.py`
> (`disabled_algorithms`). Confirm you are using the updated repo files.


In [7]:
import sys, pathlib, threading, time

# ── Locate repo root and add to path ──────────────────────────────────────────
_here = pathlib.Path().resolve()
for _p in [_here] + list(_here.parents)[:4]:
    if (_p / "lockclient.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        print("Repo root:", _p)
        break

from lockclient import LockClient, RP_client

Repo root: /Users/devesh/Desktop/Projects/RP-STCL


In [8]:
# ── Build RP_client dictionary ─────────────────────────────────────────────────

# Comment out boards that are not physically present.
RPs = {
    "Cav"  : RP_client((RP_CAV_IP,   5000), {}, mode="scan"),
    # "Lock1": RP_client((RP_LOCK1_IP, 5000), {}, mode="lock"),
    # "Mon"  : RP_client((RP_MON_IP,   5000), {}, mode="monitor"),
}

# ── Upload scripts and load settings ──────────────────────────────────────────

# LockClient.__init__ calls upload_current() on each board immediately.
print("Uploading scripts and loading settings...")
Lock = LockClient(RPs)
print("Done.")

Uploading scripts and loading settings...
Done.


In [9]:
# ── Connect (starts RunLock.py on each board via SSH) ─────────────────────────
def _wrap(fn, err):
    try:
        fn()
    except Exception as exc:
        err["exc"] = exc

def run_with_timeout(fn, timeout_s, name):
    err = {}
    t = threading.Thread(target=lambda: _wrap(fn, err), daemon=True)
    t.start()
    t.join(timeout=timeout_s)
    if t.is_alive():
        raise TimeoutError(
            "{} timed out after {}s.\n"
            "Troubleshooting:\n"
            "  1. SSH into board and run: python3 /home/jupyter/RedPitaya/RunLock.py\n"
            "  2. Check port 5000 is free: ss -tlnp | grep 5000\n"
            "  3. Kill stale processes:    pkill -f RunLock.py\n"
            "  4. Power-cycle the board if nothing else works.".format(name, timeout_s)
        )
    if "exc" in err:
        raise RuntimeError("{} failed: {}".format(name, err["exc"]))

run_with_timeout(Lock.connect_all, timeout_s=30, name="connect_all")
print("All boards connected.")

connecting...
All boards connected.


In [10]:
# ── Start PC-side event loop ───────────────────────────────────────────────────
if "stcl_thread" not in globals() or not stcl_thread.is_alive():
    stcl_thread = threading.Thread(target=Lock.start, daemon=True)
    stcl_thread.start()
    time.sleep(2)
    print("Event loop started.")
else:
    print("Event loop already running.")

# ── Connection status ──────────────────────────────────────────────────────────
for name, rp in Lock.RPs.items():
    status = "connected" if rp.connected else "DISCONNECTED"
    print("  {:6s}  {}  {}".format(name, rp.addr[0], status))

Event loop started.
  Cav     192.168.0.200  connected


---
## Phase 2: Signal Verification

**Before scanning the cavity**, verify the analog path is working by acquiring
a raw trace and plotting it.

**What to check in the plot:**
- CH1 (IN1): cavity transmission photodiode signal.
  You should see a roughly flat baseline — cavity resonance peaks appear only
  during the scan.
- CH2 (IN2): trigger square wave from OUT1 of the Cav RP.
  Should show a clean 0/1 transition.

**Wiring reminder:**
- `Cav OUT2` → piezo amplifier → cavity piezo
- `Cav OUT1` → `IN2` of all other RPs (trigger sync)
- Cavity transmission photodiode → `IN1` of all RPs

> If CH1 is flat at 0V and you expect a signal: check photodiode power,
> fiber coupling, and that the piezo amplifier is enabled.
> The scan loop has not started yet so no ramp is being output;
> this is just a single triggered capture for wiring verification.


In [11]:
import numpy as np
import matplotlib.pyplot as plt

# Single acquisition — Cav must be connected
acq = np.array(Lock.send("Cav", "acquire"))

if acq.size == 0:
    print("WARNING: No data returned. Check Cav is connected and cavity is being scanned.")
else:
    t    = acq[0]   # time axis [ms]
    ch1  = acq[1]   # IN1 — cavity transmission
    ch2  = acq[2]   # IN2 — trigger

    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    fig.suptitle("Single acquisition — Cav RP", fontsize=12)

    axes[0].plot(t, ch1, lw=0.8, color="#4fc3f7")
    axes[0].set_ylabel("IN1  [V]")
    axes[0].set_title("Cavity transmission (photodiode)")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(t, ch2, lw=0.8, color="#a5d6a7")
    axes[1].set_ylabel("IN2  [V]")
    axes[1].set_title("Trigger signal (from OUT1)")
    axes[1].set_xlabel("Time  [ms]")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("CH1  min={:.3f}  max={:.3f}  V".format(ch1.min(), ch1.max()))
    print("CH2  min={:.3f}  max={:.3f}  V".format(ch2.min(), ch2.max()))

CH1  min=-0.045  max=-0.035  V
CH2  min=-1.000  max=0.684  V


---
## Phase 3: Cavity Scan

**Goal:** Get the cavity transmission signal on screen, identify the reference
laser peaks, and set the correct ranges and lockpoint.

**Sequence:**
1. Push the Master (cavity) settings to the Cav RP.
2. Start the scan loop — this fires the triangle ramp on `OUT2` continuously.
3. *(Optional)* Open the live cavity monitor window (requires `Mon` board).
4. Observe the peaks and adjust `CAV_RANGE` / `CAV_LOCKPOINT` in Phase 0 config,
   then re-run the settings cell without restarting the scan.

**What to observe on the oscilloscope:**
- Trigger on `OUT1` (square wave), timebase ~1 ms/div.
- `OUT2` should show a triangle wave ±0.5 V at ~2.1 ms period.
- `IN1` should show the cavity transmission peaks riding on the ramp.

**Reference peak positions:**
The Master range must contain **two** peaks of the reference laser
(separated by one FSR). The ratio of distances is used to normalise errors
against scan rate fluctuations. Set `CAV_RANGE[0]` around the first peak
and `CAV_RANGE[1]` around the second.

> **Stop the scan loop** (`Lock.stop_loop("Cav")`) before moving to Phase 4.
> Never start a lock on top of a running scan loop.


In [12]:
# ── Push cavity settings ───────────────────────────────────────────────────────
Lock.update_setting("Cav", "Master", "range",      CAV_RANGE)
Lock.update_setting("Cav", "Master", "lockpoint",  CAV_LOCKPOINT)
Lock.update_setting("Cav", "Master", "enabled",    True)
Lock.update_setting("Cav", "Master", "PID",        CAV_PID)
Lock.update_setting("Cav", "Master", "dec",        CAV_DEC)

print("Cavity settings pushed:")
print("  range     :", CAV_RANGE, "ms")
print("  lockpoint :", CAV_LOCKPOINT, "ms")
print("  dec       :", CAV_DEC)
print("  PID       :", CAV_PID)

check if lockpoint is still fine
Cavity settings pushed:
  range     : [[0.25, 0.5], [1.7, 2.0]] ms
  lockpoint : 1.8 ms
  dec       : 16
  PID       : {'P': 0.0, 'I': 2.0, 'D': 0.0, 'I_val': 0, 'limit': [-0.99, 0.99]}


In [13]:
# ── Start cavity scan loop ─────────────────────────────────────────────────────

# This returns immediately — the scan runs in a background thread on the RP.

Lock.start_scan("Cav")
print("Cavity scan started on Cav ({}). OUT2 is now outputting the ramp.".format(RP_CAV_IP))
print("Use Lock.stop_loop('Cav') to stop.")

Cavity scan started on Cav (192.168.0.200). OUT2 is now outputting the ramp.
Use Lock.stop_loop('Cav') to stop.


connected to <socket.socket fd=94, family=2, type=1, proto=0, laddr=('192.168.0.115', 49418), raddr=('192.168.0.200', 5065)>


In [ ]:
# ── Optional: start live cavity monitor ───────────────────────────────────────

# Requires Mon board. Opens a matplotlib Qt5Agg window showing IN1 in real time.
# Comment out if Mon is not present.

# Lock.start_monitor("Mon")
# print("Cavity monitor started.")

# To stop: Lock.stop_monitor("Mon")


In [14]:
# ── Re-run this cell to update settings while scan is running ─────────────────

# Adjust CAV_RANGE and CAV_LOCKPOINT in Phase 0, then re-run this cell.
# No need to stop/restart the scan loop.

Lock.update_setting("Cav", "Master", "range",     CAV_RANGE)
Lock.update_setting("Cav", "Master", "lockpoint", CAV_LOCKPOINT)
print("Settings updated mid-scan.")
print("  range     :", CAV_RANGE)
print("  lockpoint :", CAV_LOCKPOINT)


check if lockpoint is still fine
Settings updated mid-scan.
  range     : [[0.25, 0.5], [1.7, 2.0]]
  lockpoint : 1.8


---
## Phase 4: Cavity Lock

**Goal:** Stabilise the cavity length by locking the reference laser peak
to `CAV_LOCKPOINT`. The PID output is fed back to the cavity piezo offset
via `OUT2` of the Cav RP.

**Before running:**
- Stop the scan loop from Phase 3: `Lock.stop_loop("Cav")`.
- Confirm the reference peak is visible and sitting near `CAV_LOCKPOINT`.
- PID gains should be conservative initially (`P=0, I=2` is a safe starting point).

**Sign check:** The `check_sign()` routine runs automatically at lock startup.
It performs 100 iterations and flips the feedback sign if the error grows.
This adds ~10 s to startup time but prevents lock instability from wrong sign.

**Verifying the lock is holding:**
- On the oscilloscope the reference peak position should stop drifting.
- The cavity PID output (`Out2` offset) will fluctuate within `limit` range.
- Watch `Lock.RPs["Cav"].settings` or the error monitor (Phase 6) for confirmation.

> **If the lock immediately rails** (PID hits `±0.99 V` and stays there):
> the sign is wrong or the lockpoint is outside the range. Stop with
> `Lock.stop_loop("Cav")`, check the peak position, and retry.


In [15]:
# ── Stop scan loop (must be stopped before locking) ───────────────────────────
if Lock.RPs["Cav"].loop_running:
    Lock.stop_loop("Cav")
    time.sleep(1.0)
    print("Scan loop stopped.")
else:
    print("Scan loop was not running.")

Scan loop stopped.


In [16]:
# ── Push final cavity lock settings ───────────────────────────────────────────
Lock.update_setting("Cav", "Master", "range",     CAV_RANGE)
Lock.update_setting("Cav", "Master", "lockpoint", CAV_LOCKPOINT)
Lock.update_setting("Cav", "Master", "PID",       CAV_PID)
print("Cavity lock settings confirmed.")

check if lockpoint is still fine
Cavity lock settings confirmed.


In [ ]:
# ── Start cavity lock ──────────────────────────────────────────────────────────
# Runs in background thread. The RP enters the PID loop immediately.
Lock.start_lock("Cav")
print("Cavity lock started.")
print("Monitor PID output with:  Lock.RPs['Cav'].loop_running")
print("Stop with:                Lock.stop_loop('Cav')")

---
## Phase 5: Laser Lock

**Requires:** `Lock1` board connected and `Cav` lock running stably.

**Goal:** Lock one or two slave lasers by feeding back to their current
modulation inputs via `Lock1 OUT1` (Slave1) and `Lock1 OUT2` (Slave2).

**Wiring:**
- `Lock1 IN1` ← cavity transmission photodiode (same signal as Cav IN1)
- `Lock1 IN2` ← trigger from `Cav OUT1`
- `Lock1 OUT1` → Laser A current modulation input
- `Lock1 OUT2` → Laser B current modulation input (if Slave2 enabled)

**Setting ranges:** Each slave range must contain exactly one peak of the
corresponding slave laser. The position is measured *relative to the first
reference peak*, so small cavity drifts do not affect the error signal.

**PID tuning sequence:**
1. Start with `I` only (`P=0, I=0.5`). Watch the peak stabilise.
2. Slowly increase `I` until the peak holds position under perturbation.
3. Add `P` only if faster transient response is needed.
4. Leave `D=0` unless you have a specific reason (adds noise sensitivity).

> **If Slave1/Slave2 peaks are not visible:** the laser is not coupled into
> the cavity, or the range does not contain the peak. Adjust `SL1_RANGE` /
> `SL2_RANGE` in Phase 0 and re-run the settings cell.


In [ ]:
# ── Check Lock1 is available ───────────────────────────────────────────────────
if "Lock1" not in Lock.RPs:
    print("Lock1 not in RPs dict — skipping laser lock phase.")
    print("Add Lock1 to the BOARDS config in Phase 0 and restart from Phase 1.")
else:
    print("Lock1 found:", Lock.RPs["Lock1"].addr[0])
    print("Connected  :", Lock.RPs["Lock1"].connected)

In [ ]:
# ── Push Slave1 settings ───────────────────────────────────────────────────────
if "Lock1" in Lock.RPs:
    Lock.update_setting("Lock1", "Slave1", "label",     SL1_LABEL)
    Lock.update_setting("Lock1", "Slave1", "range",     SL1_RANGE)
    Lock.update_setting("Lock1", "Slave1", "lockpoint", SL1_LOCKPOINT)
    Lock.update_setting("Lock1", "Slave1", "enabled",   SL1_ENABLED)
    Lock.update_setting("Lock1", "Slave1", "PID",       SL1_PID)
    print("Slave1 ({}) settings pushed:".format(SL1_LABEL))
    print("  range     :", SL1_RANGE, "ms")
    print("  lockpoint :", SL1_LOCKPOINT, "ms")
    print("  enabled   :", SL1_ENABLED)
    print("  PID       :", SL1_PID)

In [ ]:
# ── Push Slave2 settings (only if SL2_ENABLED = True) ─────────────────────────
if "Lock1" in Lock.RPs:
    Lock.update_setting("Lock1", "Slave2", "label",     SL2_LABEL)
    Lock.update_setting("Lock1", "Slave2", "range",     SL2_RANGE)
    Lock.update_setting("Lock1", "Slave2", "lockpoint", SL2_LOCKPOINT)
    Lock.update_setting("Lock1", "Slave2", "enabled",   SL2_ENABLED)
    Lock.update_setting("Lock1", "Slave2", "PID",       SL2_PID)
    print("Slave2 ({}) settings pushed  (enabled={})".format(SL2_LABEL, SL2_ENABLED))

In [ ]:
# ── Start laser lock ───────────────────────────────────────────────────────────
if "Lock1" in Lock.RPs:
    Lock.start_lock("Lock1")
    print("Laser lock started on Lock1.")
    print("Stop with: Lock.stop_loop('Lock1')")

---
## Phase 6: Error Monitor

**Requires:** `Mon` board connected, cavity lock running.

Opens a live matplotlib window showing the **frequency error** of each
locked laser channel in MHz over time. Data is recorded in memory and
can be saved to a JSON file.

The error is scaled by `FSR` (free spectral range in MHz, default 906 MHz)
so the y-axis reads directly in MHz.

**What to watch:**
- Errors should sit near 0 MHz with small fluctuations (< 1 MHz when well locked).
- Slow drift means the PID integral gain is too low.
- Fast oscillations mean the proportional gain is too high.
- A sudden jump means the lock broke — check the peak position.

> **Mon board not present?** You can still read instantaneous errors without
> the monitor board using `Lock.send("Cav", "acquire_errs")` — this uses
> the Cav board itself for error readout (less dedicated but functional).


In [ ]:
# ── Start cavity signal monitor (Mon board) ────────────────────────────────────
if "Mon" in Lock.RPs:
    Lock.start_monitor("Mon")
    print("Cavity monitor started.")
else:
    print("Mon board not present — skipping cavity monitor.")
    print("You can still watch errors with the error monitor below.")

In [ ]:
# ── Start error monitor (Mon board) ───────────────────────────────────────────

# tmin: minimum time between error samples in seconds (default 10 ms).
# Opens a live plot window. Runs until Lock.stop_monitor("Mon") is called.

if "Mon" in Lock.RPs:
    Lock.stop_monitor("Mon")          # stop cavity monitor first if running
    time.sleep(0.5)
    Lock.start_error_monitor("Mon", tmin=20e-3)
    print("Error monitor started. Close the plot window or run the next cell to stop.")
else:
    print("Mon board not present — skipping error monitor.")

In [ ]:
# ── Save error data ────────────────────────────────────────────────────────────
# Saves the recorded errors to a JSON file. Run while monitor is still open.
# The file will contain time axis + per-channel error arrays.

# import time as _t
# filename = "lock_errors_{}".format(int(_t.time()))
# if "Mon" in Lock.RPs:
#     Lock.monitors["Mon"]["queue_err"].put(("save", filename))
#     print("Saving errors to {}.json".format(filename))

---
## Phase 7: Safe Shutdown

**Always shut down in this order:**
1. Stop error / cavity monitors first.
2. Stop laser lock loops (`Lock1`) before stopping the cavity scan/lock.
3. Stop cavity loop last — the cavity RP generates the trigger for all others.
4. Disconnect all RPs (closes socket servers on boards).
5. Stop the PC-side event loop.

Skipping this order can leave boards with stale processes holding ports
5000 / 5065, requiring a manual `pkill` on the board before the next session.

> `Lock.close()` performs all steps above in the correct order automatically.
> Use the individual cells below only when you need fine-grained control
> (e.g. stopping one laser lock while keeping the cavity lock running).


In [17]:
# ── Stop monitors ──────────────────────────────────────────────────────────────
if "Mon" in Lock.RPs:
    Lock.stop_monitor("Mon")
    time.sleep(0.5)
    print("Monitor stopped.")

In [18]:
# ── Stop laser lock ────────────────────────────────────────────────────────────
if "Lock1" in Lock.RPs and Lock.RPs["Lock1"].loop_running:
    Lock.stop_loop("Lock1")
    time.sleep(0.5)
    print("Laser lock stopped.")

In [14]:
# ── Stop cavity lock / scan ────────────────────────────────────────────────────
if Lock.RPs["Cav"].loop_running:
    Lock.stop_loop("Cav")
    time.sleep(0.5)
    print("Cavity loop stopped.")

Cavity loop stopped.


In [15]:
# ── Full clean shutdown ────────────────────────────────────────────────────────
# Stops all loops, disconnects all boards, stops the event loop.
# Run this at the end of every session.

Lock.close()
print("All boards disconnected. Session closed.")
print()
print("Board port status after shutdown:")
print("  Ports 5000 and 5065 should now be free on all boards.")
cmd = 'ss -tlnp | grep -E "5000|5065"'
print(f" Verify with:  ssh root@<IP> {cmd}")


All boards disconnected. Session closed.

Board port status after shutdown:
  Ports 5000 and 5065 should now be free on all boards.
 Verify with:  ssh root@<IP> ss -tlnp | grep -E "5000|5065"
